# Decade-wise split in Fields

### 1. CSV Generation

#### We will generate CSVs that contain the field's raw counts and percentages

In [ ]:
import pandas as pd

# Load the awards dataset
file_path = '../datasets/cleaned_datasets/national_awards_cleaned.csv'
df = pd.read_csv(file_path)

df_decade = df.dropna(subset=['Year']).copy()
df_decade['Decade'] = (df_decade['Year'] // 10 * 10).astype(int).astype(str) + 's'

# Decade Counts CSV
decade_counts = pd.crosstab(df_decade['Field'], df_decade['Decade'])
decade_counts.to_csv('./field_counts_by_decade.csv', index_label='Field')
print('Exported: field_counts_by_decade.csv')

# Decade Percentages CSV (%)
decade_pct = (pd.crosstab(df_decade['Field'], df_decade['Decade'], normalize='columns') * 100).round(2)
decade_pct.to_csv('field_percentages_by_decade.csv', index_label='Field')
print('Exported: field_percentages_by_decade.csv')

### 2. Graphs 

#### We will generate graphs to provide a visualisation of the trends observed

A. Raw counts

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sns.set_theme(style='whitegrid')

df = pd.read_csv('field_counts_by_decade.csv')
decade_cols = [c for c in df.columns if c != 'Field']

df_melted = df.melt(id_vars='Field', value_vars=decade_cols,
                    var_name='Decade', value_name='Recipients')

# 1. Stacked bar
ax = df.set_index('Field')[decade_cols].T.plot(
    kind='bar', stacked=True, figsize=(12, 6),
    color=sns.color_palette('tab20', n_colors=len(df)))
ax.set_title('Awards by Field Across Decades')
ax.set_xlabel('Decade')
ax.set_ylabel('Award Count')
ax.legend(title='Field', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

# 2. Line chart
plt.figure(figsize=(12, 6))
sns.lineplot(data=df_melted, x='Decade', y='Recipients', hue='Field',
             marker='o', palette='tab20')
plt.title('Award Trends by Field')
plt.legend(title='Field', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.show()

# 3. Heatmap
plt.figure(figsize=(10, max(5, len(df) * 0.4)))
sns.heatmap(df.set_index('Field')[decade_cols], annot=True, fmt='.0f',
            cmap='Blues', cbar_kws={'label': 'Recipients'})
plt.title('Decade vs Field Heatmap')
plt.xlabel('Decade')
plt.ylabel('Field')
plt.tight_layout()
plt.show()

B. Percentage Data

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sns.set_theme(style='whitegrid')

df = pd.read_csv('field_percentages_by_decade.csv')
decade_cols = [c for c in df.columns if c != 'Field']

n = len(decade_cols)
cols = min(n, 3)
rows = (n + cols - 1) // cols

fig, axes = plt.subplots(rows, cols, figsize=(6 * cols, 6 * rows))
axes = axes.flatten() if n > 1 else [axes]

colors = sns.color_palette('tab20', len(df))

def pct_fmt(p):
    return f'{p:.1f}%' if p >= 4 else ''   # hide small slices' labels

for ax, decade in zip(axes, decade_cols):
    wedges, _, autotexts = ax.pie(
        df[decade],
        autopct=pct_fmt,
        pctdistance=0.8,
        startangle=90,
        colors=colors,
        wedgeprops={'edgecolor': 'white'},
        textprops={'fontsize': 9},
    )
    ax.set_title(decade)

for ax in axes[n:]:
    ax.axis('off')

# One shared legend instead of per-slice labels
fig.legend(wedges, df['Field'], loc='center left',
           bbox_to_anchor=(1.0, 0.5), title='Field')

plt.tight_layout()
plt.show()

### 3. Chi - square test

#### Now, using the standard chi-square test of homogeneity, we will check the hypotheses

In [ ]:
from scipy.stats import chi2_contingency

counts = pd.read_csv('field_counts_by_decade.csv', index_col='Field')

chi2, p, dof, expected = chi2_contingency(counts)
print(f'chi2={chi2:.2f}, dof={dof}, p={p:.4g}')

expected = pd.DataFrame(expected, index=counts.index, columns=counts.columns)
print(f'cells with expected < 5: {(expected < 5).sum().sum() / expected.size:.0%}')

resid = (counts - expected) / expected ** 0.5   # |value| > 2 is notable
plt.figure(figsize=(10, max(5, len(counts) * 0.4)))
sns.heatmap(resid, center=0, cmap='RdBu_r', annot=True, fmt='.1f',
            cbar_kws={'label': 'Standardized residual'})
plt.title('Over/under-representation by field and decade')
plt.tight_layout()
plt.show()

But, if more than about 20% of cells have expected counts below 5, the chi-square approximation is unreliable.

To fix this, we will merge some of the fields with a smaller representation

We will also use a Monte Carlo permutuation on the original table

Here's how we run the simulation:
1. We take every award and its (field, decade) pair. Then, we compute the chi-square statistic on that real data. This is our observed value.
2. Then we shuffle the decade labels randomly among the awards, so each award keeps its field but gets a random decade. Now we the chi-square statistic.
3. Repeating this 2000 times gives us 2000 different chi square statistics.
The p-value is the fraction of shuffled values that are as large as or larger than our observed one. If the real data beats nearly all of them, the pattern is very unlikely to be chance.

The standard chi-square p-value relies on an approximation that only works when expected counts are not too small. Our table had many sparse fields across many decades, so that approximation was in doubt. The permutation test makes no such assumption, so it checks that our conclusion holds regardless.

In [ ]:
import random

counts = pd.read_csv('field_counts_by_decade.csv', index_col='Field')

# Merge rare fields into "Other" until all expected counts >= 5
def min_expected(df):
    return df.sum(axis=1) * df.sum().min() / df.sum().sum()

merged = counts.copy()
rare = merged.index[min_expected(merged) < 5]
merged.loc['Other'] = merged.loc[rare].sum()
merged = merged.drop(rare)

while min_expected(merged).min() < 5 and len(merged) > 2:
    smallest = merged.drop('Other').sum(axis=1).idxmin()
    merged.loc['Other'] += merged.loc[smallest]
    merged = merged.drop(smallest)

chi2, p, dof, expected = chi2_contingency(merged)
print(f'Merged chi-square: chi2={chi2:.2f}, dof={dof}, p={p:.4g}')
print('Fields kept:', list(merged.index))

# Monte Carlo permutation test on the original table
long = counts.stack().rename('n').reset_index()
long.columns = ['Field', 'Decade', 'n']
fields = long['Field'].repeat(long['n']).tolist()
decades = long['Decade'].repeat(long['n']).tolist()

def chi2_stat(f, d):
    obs = pd.crosstab(pd.Series(f), pd.Series(d))
    exp = obs.sum(axis=1).to_frame() @ obs.sum().to_frame().T / obs.sum().sum()
    return (((obs - exp) ** 2) / exp).sum().sum()

observed = chi2_stat(fields, decades)
B = 2000
extreme = 0
for _ in range(B):
    random.shuffle(decades)
    if chi2_stat(fields, decades) >= observed:
        extreme += 1

p_mc = (extreme + 1) / (B + 1)
print(f'Monte Carlo: chi2={observed:.2f}, p={p_mc:.4f}')

# Residual heatmap on the merged table
expected = pd.DataFrame(expected, index=merged.index, columns=merged.columns)
resid = (merged - expected) / expected ** 0.5

plt.figure(figsize=(10, 6))
sns.heatmap(resid, center=0, cmap='RdBu_r', annot=True, fmt='.1f',
            cbar_kws={'label': 'Standardized residual'})
plt.title('Over/under-representation by field and decade')
plt.tight_layout()
plt.show()

### 4. Conclusion

The heatmaps show us how the relative representation of the awards themselves have changed over the decades.

1. There has been less of an emphasis on Civil Services compared to the 70s
2. Science and Engineering has seen lesser proportional representation compared to the 2000s
3. Spiritualism, agriculture, journalism, archeology, social work and grassroots innovation have seen a boom in the 2010s and 2020s
4. Sports awardees have had an increased representation in the past 2 decades
5. Literature and Education and Medicine have had more or less even representation throughout

Therefore, H0 is rejected, and the data supports H1.